# 06 Pre-LN 与 Post-LN Transformer 有什么训练差异？

## 面试回答主线

Pre-LN 在子层之前做归一化，残差支路是更直接的恒等路径；Post-LN 则在残差相加后归一化。两者并非谁绝对更强：Post-LN 在某些配方下可达到很高质量，但深层训练更依赖 warmup、初始化和残差尺度；Pre-LN 往往更容易从头稳定优化。面试回答必须区分前向表达与反向路径，并强调最终归一化、学习率和深度会改变结论。本实验手写小型残差堆叠，测量首层梯度范数和无归一化时的激活漂移。它是结构诊断，不是语言建模基准。

**核心公式：** Pre-LN：$x_{l+1}=x_l+F(\operatorname{LN}(x_l))$；Post-LN：$x_{l+1}=\operatorname{LN}(x_l+F(x_l))$。Pre-LN 的残差恒等支路使梯度可直接跨层传播。

本 Notebook 将依次展示业务输入、可比较基线、手写核心机制、中间量、失败与修复；所有数值都是确定性的教学实验。


## 真实案例

场景是支付与账户安全客服系统：模型要把工单分成“高风险需优先人工处理”和“常规处理”。三个输入特征分别表示资金风险线索、登录/身份线索和售后/账单线索。数据为人工构造的脱敏离线事件，字段结构模拟真实工单，不可外推为生产表现。


In [1]:
import math  # 导入数学函数以实现尺度公式。
import warnings  # 导入警告控制模块以保持教学输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的非教学弃用警告。
import torch  # 导入 PyTorch 张量和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(17)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验波动。
samples = [  # 构造脱敏客服工单的真实语义样本。
    {'ticket': '支付重复扣款，要求退款', 'features': [1.0, 0.0, 1.0], 'label': 1},  # 高风险退款工单。
    {'ticket': '登录验证码收不到', 'features': [0.0, 1.0, 0.0], 'label': 0},  # 普通技术支持工单。
    {'ticket': '账户出现陌生转账', 'features': [1.0, 0.0, 0.0], 'label': 1},  # 高风险资金安全工单。
    {'ticket': '如何修改收货地址', 'features': [0.0, 0.0, 1.0], 'label': 0},  # 普通售后咨询工单。
    {'ticket': '银行卡被盗刷请冻结', 'features': [1.0, 1.0, 0.0], 'label': 1},  # 高风险且紧急的工单。
    {'ticket': '发票抬头需要更正', 'features': [0.0, 1.0, 1.0], 'label': 0},  # 低风险但需要人工处理的工单。
]  # 结束教学样本定义。
features = torch.tensor([row['features'] for row in samples], dtype=torch.float32)  # 将可读字段转为模型输入张量。
labels = torch.tensor([row['label'] for row in samples], dtype=torch.long)  # 将风险标签转为分类目标。
print('教学实验：脱敏客服工单，不代表线上规模或泛化收益。')  # 明确实验边界。
for row in samples:  # 逐条展示输入样本而不是隐藏在张量中。
    print(f"标签={row['label']} | 特征={row['features']} | 工单={row['ticket']}")  # 输出原始业务语义。
print(f'输入张量形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状和目标。


教学实验：脱敏客服工单，不代表线上规模或泛化收益。
标签=1 | 特征=[1.0, 0.0, 1.0] | 工单=支付重复扣款，要求退款
标签=0 | 特征=[0.0, 1.0, 0.0] | 工单=登录验证码收不到
标签=1 | 特征=[1.0, 0.0, 0.0] | 工单=账户出现陌生转账
标签=0 | 特征=[0.0, 0.0, 1.0] | 工单=如何修改收货地址
标签=1 | 特征=[1.0, 1.0, 0.0] | 工单=银行卡被盗刷请冻结
标签=0 | 特征=[0.0, 1.0, 1.0] | 工单=发票抬头需要更正
输入张量形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先看最简单的对照。基线与核心方案使用完全相同的样本、标签和指标，避免把数据变化误认为算法收益。


In [2]:
sequence = features.unsqueeze(1).repeat(1, 2, 1)  # 将工单特征复制成两个 token 的短序列。
target_sequence = torch.zeros_like(sequence)  # 创建稳定的零目标以测量反向传播。
class ManualLayerNorm(nn.Module):  # 手写按最后维归一化层。
    def __init__(self, width):  # 创建缩放与平移参数。
        super().__init__()  # 初始化模块父类。
        self.scale = nn.Parameter(torch.ones(width))  # 初始化缩放参数。
        self.bias = nn.Parameter(torch.zeros(width))  # 初始化偏移参数。
    def forward(self, value):  # 按 token 最后一维执行 LayerNorm。
        mean = value.mean(dim=-1, keepdim=True)  # 计算每个 token 的均值。
        variance = (value - mean).pow(2).mean(dim=-1, keepdim=True)  # 计算每个 token 的方差。
        normalized = (value - mean) / torch.sqrt(variance + 1e-5)  # 执行中心化和标准化。
        return normalized * self.scale + self.bias  # 返回仿射变换后的结果。
class ResidualBlock(nn.Module):  # 明确实现一个可切换 norm 位置的残差块。
    def __init__(self, width, pre_norm):  # 保存宽度和 norm 位置。
        super().__init__()  # 初始化模块父类。
        self.pre_norm = pre_norm  # 记录是否采用 Pre-LN。
        self.norm = ManualLayerNorm(width)  # 创建手写 LayerNorm。
        self.weight = nn.Parameter(torch.randn(width, width) / math.sqrt(width))  # 创建子层投影矩阵。
    def forward(self, value):  # 明确写出残差和归一化顺序。
        if self.pre_norm:  # 处理 Pre-LN 分支。
            return value + torch.tanh(self.norm(value) @ self.weight)  # 先归一化后进入子层并保留恒等残差。
        return self.norm(value + torch.tanh(value @ self.weight))  # 先残差相加再做 Post-LN。
class ResidualStack(nn.Module):  # 用 ModuleList 组合多个手写残差块。
    def __init__(self, depth, pre_norm):  # 建立指定深度的堆叠。
        super().__init__()  # 初始化模块父类。
        self.blocks = nn.ModuleList([ResidualBlock(3, pre_norm) for _ in range(depth)])  # 创建每一层残差块。
    def forward(self, value):  # 逐层传播短序列。
        for block in self.blocks:  # 遍历各层残差块。
            value = block(value)  # 更新当前隐藏状态。
        return value  # 返回最终序列表示。
post_model = ResidualStack(10, False)  # 创建 Post-LN 基线模型。
post_loss = (post_model(sequence) - target_sequence).pow(2).mean()  # 计算 Post-LN 的回归损失。
post_loss.backward()  # 反向传播以观察首层梯度。
baseline_metric = float(post_model.blocks[0].weight.grad.norm())  # 记录 Post-LN 首层梯度范数。
print(f'Post-LN：loss={post_loss.item():.4f}，首层梯度范数={baseline_metric:.6f}')  # 展示基线梯度路径。


Post-LN：loss=1.0000，首层梯度范数=0.000005


## 手写核心实现与中间量

以下实现刻意保留关键矩阵、梯度、范数或调度状态，目的是让面试时能解释“它到底改变了哪一个量”。


In [3]:
pre_model = ResidualStack(10, True)  # 创建结构相同的 Pre-LN 模型。
pre_output = pre_model(sequence)  # 运行真实前向传播。
pre_loss = (pre_output - target_sequence).pow(2).mean()  # 计算同一目标上的损失。
pre_loss.backward()  # 反向传播得到 Pre-LN 梯度。
pre_grad = float(pre_model.blocks[0].weight.grad.norm())  # 读取 Pre-LN 首层梯度范数。
pre_activation_rms = float(pre_output.pow(2).mean().sqrt())  # 记录最终激活 RMS。
core_metric = pre_grad  # 保存核心梯度指标。
print(f'Pre-LN：loss={pre_loss.item():.4f}，首层梯度范数={pre_grad:.6f}，最终激活 RMS={pre_activation_rms:.4f}')  # 输出核心中间量。


Pre-LN：loss=10.8533，首层梯度范数=4.501894，最终激活 RMS=3.2944


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 汇总同一指标口径下的可读结果表。
for name, metric in comparison_rows:  # 逐行输出基线与核心方案。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示结果表而不是只保留变量名。


Baseline | 指标=0.000005
核心机制     | 指标=4.501894


## 结果解读

请把基线和核心输出看成机制证据而非榜单。这里的指标只在同一受控工单集上可比：核心方案展示了 **Pre-LN/Post-LN** 的关键状态与更新路径。生产需统一 checkpoint 中 norm 的位置、最终 norm、残差分支精度和 fused kernel；不能只替换一行 LayerNorm 就加载旧权重。

## 失败案例

接下来故意破坏一个必要条件，再用明确的门禁、尺度或统计口径修复它。这样可以避免“代码能跑”却不知道为什么线上会失效。


In [5]:
class NoNormStack(nn.Module):  # 构造不带归一化的失败对照。
    def __init__(self, depth):  # 创建多层投影矩阵。
        super().__init__()  # 初始化模块父类。
        self.weights = nn.ParameterList([nn.Parameter(torch.randn(3, 3) * 1.4) for _ in range(depth)])  # 故意使用偏大的初始化。
    def forward(self, value):  # 重复执行无 norm 的残差更新。
        for weight in self.weights:  # 遍历每个投影矩阵。
            value = value + value @ weight  # 故意去掉激活限幅以放大无归一化残差累积。
        return value  # 返回不受控的隐藏状态。
no_norm_rms = float(NoNormStack(10)(sequence).pow(2).mean().sqrt())  # 测量无归一化的激活尺度。
failure_metric = no_norm_rms  # 保存失败的激活 RMS。
fix_metric = pre_activation_rms  # 使用 Pre-LN 的激活 RMS 作为修复对照。
print(f'失败：无归一化激活 RMS={failure_metric:.3f}；修复：Pre-LN 激活 RMS={fix_metric:.3f}')  # 展示尺度控制效果。


失败：无归一化激活 RMS=912.485；修复：Pre-LN 激活 RMS=3.294


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产需统一 checkpoint 中 norm 的位置、最终 norm、残差分支精度和 fused kernel；不能只替换一行 LayerNorm 就加载旧权重。

**常见坑：** 用浅层单 batch 的 loss 宣称某种 norm 一定更好，或者漏掉 final norm 与 warmup 的联合作用。

**延伸追问：** 为什么深度增加时 Post-LN 常需要更谨慎的 warmup？RMSNorm 替代 LayerNorm 后，Pre/Post 的稳定性假设有哪些变化？

## 生产差距

本实验只有 6 条脱敏离线工单、CPU 和 FP32，省略了大规模 token packing、数据并行、混合精度、checkpoint、指标告警和灰度回滚。生产实现应替换为真实数据管道与观测系统，并用验证集和线上安全指标决定是否发布。


In [6]:
assert baseline_metric > 0.0  # 验证 Post-LN 首层仍接收到可测梯度。
assert core_metric > 0.0  # 验证 Pre-LN 首层仍接收到可测梯度。
assert torch.isfinite(pre_output).all()  # 验证 Pre-LN 前向输出没有数值异常。
assert failure_metric > fix_metric  # 验证归一化缓解了本受控实验的激活累积。
